# TTS benchmark: Maya 2 Native vs IndicF5 (cloned voice, English / Hindi / Hinglish)
Measures **time to first audio** (keep a model only if under 1 s) and saves WAVs to listen to.

1. **Runtime → Change runtime type → T4 GPU** (L4/A100 is better), Save.
2. Add your Hugging Face token in the key icon (Secrets) as `HF_TOKEN`, and turn **Notebook access** on. IndicF5 is gated, so accept its licence on Hugging Face first.
3. Upload `stt-test/ref/karan_ref_hi.wav` when cell 3 asks (your own voice, 4.4 s).
4. Run all cells. The last cell downloads `tts-results.zip`: send it back.

Use only voices whose owner agreed. This is a test, not for client calls.

In [ ]:
!nvidia-smi | head -15

In [ ]:
!pip -q install "maya3[server] @ git+https://github.com/MayaResearch/maya2-native" git+https://github.com/ai4bharat/IndicF5.git soundfile


In [ ]:
from google.colab import files, userdata
import os
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
up = files.upload()   # choose karan_ref_hi.wav
assert 'karan_ref_hi.wav' in up
REF = '/content/karan_ref_hi.wav'
REF_TEXT = 'नमस्ते और बिज़्फी में कॉल करने के लिए आपका धन्यवाद'
TESTS = {
 'en': ('Hello, and thank you for calling Bizfy. Our plans start at 4,999 rupees per month, and you can cancel at any time.', 'en'),
 'hi': ('नमस्ते, बिज़्फी में कॉल करने के लिए धन्यवाद। हमारी योजनाएँ चार हज़ार नौ सौ निन्यानवे रुपये प्रति माह से शुरू होती हैं।', 'hi'),
 'hinglish': ('Hello, Bizfy mein call karne ke liye thank you. Hamare plans 4,999 rupees per month se start hote hain, aur aap kabhi bhi cancel kar sakte hain.', 'hi'),
}
os.makedirs('/content/out', exist_ok=True)
RESULTS = []

## Maya 2 Native (streaming, so first-chunk time is the real first-audio time)

In [ ]:
import time, numpy as np, soundfile as sf
from maya3 import Maya3TTS
tts = Maya3TTS.from_pretrained('maya-research/maya2-native', max_seats=1)
tts.register_voice('karan', ref_audio=REF, ref_text=REF_TEXT, lang='hi')
list(tts.stream('warm up', lang='en', voice='karan', normalize=True))   # warm-up, not timed
for name, (text, lang) in TESTS.items():
    for run in range(3):
        t0 = time.time(); first = None; chunks = []
        for c in tts.stream(text, lang=lang, voice='karan', normalize=True):
            if first is None: first = time.time() - t0
            chunks.append(c)
        total = time.time() - t0
        audio = np.concatenate(chunks).astype('float32') / 32768.0
        if run == 2: sf.write(f'/content/out/maya2_{name}.wav', audio, 24000)
        RESULTS.append(dict(model='maya2', clip=name, run=run, first_audio_s=round(first, 3), total_s=round(total, 2), audio_s=round(len(audio)/24000, 1)))
        print(RESULTS[-1])
tts.close()
import gc, torch; del tts; gc.collect(); torch.cuda.empty_cache()

## IndicF5 (not streaming: the whole clip is made first, so total time is the time to first audio)

In [ ]:
from transformers import AutoModel
f5 = AutoModel.from_pretrained('ai4bharat/IndicF5', trust_remote_code=True).to('cuda')
f5(TESTS['hi'][0], ref_audio_path=REF, ref_text=REF_TEXT)   # warm-up, not timed
for name, (text, lang) in TESTS.items():
    for run in range(3):
        t0 = time.time(); a = f5(text, ref_audio_path=REF, ref_text=REF_TEXT); total = time.time() - t0
        if a.dtype == np.int16: a = a.astype(np.float32) / 32768.0
        a = np.array(a, dtype=np.float32)
        if run == 2: sf.write(f'/content/out/indicf5_{name}.wav', a, 24000)
        RESULTS.append(dict(model='indicf5', clip=name, run=run, first_audio_s=round(total, 3), total_s=round(total, 2), audio_s=round(len(a)/24000, 1)))
        print(RESULTS[-1])

In [ ]:
import json, pandas as pd
df = pd.DataFrame(RESULTS); json.dump(RESULTS, open('/content/out/results.json', 'w'), indent=1)
s = df.groupby(['model', 'clip']).first_audio_s.median().unstack(); s['under_1s'] = (s.max(axis=1) < 1.0)
print(s)
!cd /content && zip -qr tts-results.zip out
files.download('/content/tts-results.zip')